# Entendimento inicial do dataset de voos

Este notebook explora o CSV derivado do Voo Regular Ativo (VRA) da ANAC. Ele não treina modelos. O objetivo é verificar estrutura, cobertura, alvos, valores ausentes, distribuição de atrasos e outliers antes de definir as variáveis preditoras.

Para gerar a base local, execute `py scripts/preparar_dados_v2.py` na raiz do projeto. O CSV é local e não é versionado.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'voos_vra_derivados.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {DATA_PATH}. Execute o script de preparação primeiro.')
DATA_PATH

In [ ]:
date_columns = ['partida_prevista', 'partida_real', 'chegada_prevista', 'chegada_real']
df = pd.read_csv(DATA_PATH, parse_dates=date_columns, low_memory=False)
print('Dimensões:', df.shape)
df.head(3)

## 1. Schema e qualidade básica

As colunas de horários reais e situações operacionais são rótulos ou informações posteriores. Elas não devem entrar como preditoras no primeiro experimento.

In [ ]:
schema = pd.DataFrame({'tipo': df.dtypes.astype(str), 'ausentes': df.isna().sum(), 'percentual_ausente': (100 * df.isna().mean()).round(2)})
schema

In [ ]:
print('Duplicidades:', int(df.duplicated().sum()))
print('Arquivos de origem:', df['arquivo_origem'].value_counts().sort_index().to_dict())
print('Situação do voo:')
display(df['situacao_voo'].value_counts(dropna=False).to_frame('quantidade'))
print('Situação da chegada:')
display(df['situacao_chegada'].value_counts(dropna=False).to_frame('quantidade'))

## 2. Alvos iniciais

`cancelado` é um alvo separado. Para atraso de chegada, usamos somente voos realizados com horário previsto e real válidos. O primeiro alvo recomendado é `atraso_chegada_15m`, uma classificação binária.

In [ ]:
target_summary = pd.DataFrame({
    'quantidade': [df['cancelado'].sum(), df['realizado'].sum(), df['atraso_chegada_15m'].notna().sum(), df['atraso_chegada_15m'].sum()],
}, index=['cancelados', 'realizados', 'alvos_de_atraso_observáveis', 'chegadas_15m_ou_mais_atrasadas'])
target_summary

In [ ]:
realizados = df[df['realizado']].copy()
realizados[['atraso_partida_min', 'atraso_chegada_min']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)

## 3. Distribuição e outliers

A amostra já revelou registros com atrasos superiores a 24 horas e casos acima de 30 dias. Esses valores não devem ser removidos automaticamente. O alvo binário é menos sensível a esse problema; uma regressão em minutos exigirá uma regra de tratamento definida antes da avaliação.

In [ ]:
outliers = realizados[realizados['atraso_chegada_min'] > 24 * 60].sort_values('atraso_chegada_min', ascending=False)
print('Atrasos de chegada acima de 24 horas:', len(outliers))
outliers[['arquivo_origem', 'companhia_icao', 'numero_voo', 'origem_icao', 'destino_icao', 'chegada_prevista', 'chegada_real', 'atraso_chegada_min', 'situacao_chegada']].head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
realizados['atraso_chegada_min'].clip(-60, 360).plot.hist(bins=60, ax=axes[0], title='Atraso de chegada (limitado a -60 a 360 min)')
realizados['atraso_chegada_15m'].value_counts().sort_index().plot.bar(ax=axes[1], title='Alvo: chegada com atraso >= 15 min')
axes[0].set_xlabel('Minutos')
axes[1].set_xlabel('False / True')
plt.tight_layout()

## 4. Padrões por tempo, companhia e aeroporto

Estas tabelas são descritivas. Elas não demonstram causalidade e não devem ser usadas diretamente como estimativas do desempenho futuro sem uma divisão temporal.

In [ ]:
realizados['mes_previsto'] = realizados['partida_prevista'].dt.to_period('M').astype(str)
realizados['dia_semana'] = realizados['partida_prevista'].dt.day_name()
realizados['hora_prevista'] = realizados['partida_prevista'].dt.hour
monthly = realizados.groupby('mes_previsto').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean'), mediana_atraso_min=('atraso_chegada_min', 'median')).reset_index()
monthly

In [ ]:
by_carrier = realizados.groupby('companhia_icao').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean'), mediana_atraso_min=('atraso_chegada_min', 'median')).query('voos >= 100').sort_values('taxa_atraso_15m', ascending=False)
by_carrier.head(20)

In [ ]:
by_origin = realizados.groupby('origem_icao').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean')).query('voos >= 100').sort_values('taxa_atraso_15m', ascending=False)
by_hour = realizados.groupby('hora_prevista').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean')).reset_index()
display(by_origin.head(20))
display(by_hour)

## Conclusão provisória

O dataset é grande em número de voos, mas ainda cobre apenas dois meses. O próximo notebook deve construir variáveis preditoras disponíveis antes do voo, definir a janela temporal de treino/validação/teste e comparar baselines simples. As taxas acima são exploratórias e não representam previsão operacional.